### Qwen 0.5B

In [7]:
import yaml
import copy
import os

### 10%
# compression_factors = [
#     1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,
#     1.02, 1.25, 1.34, 1.22, 1.32, 1.34, 1.14, 1.08,
#     1.03, 1.21, 1.51, 1.24, 1.13, 1.26, 1.19, 1.0
# ]

### 20%
# compression_factors = [1.  , 1.  , 1.26, 1.21, 1.2 , 1.29, 1.36, 1.43, 1.47, 1.8 , 1.92, 1.75,
#  1.9 , 1.92, 1.64, 1.55, 1.47, 1.74, 2.17, 1.79, 1.63, 1.8 , 1.71, 1.  ]

### 30%
# compression_factors = [1.  , 1.  , 1.26, 1.21, 1.2 , 1.29, 1.36, 1.43, 1.47, 1.8 , 1.92, 1.75,
#   1.9 , 1.92, 1.64, 1.55, 1.47, 1.74, 2.17, 1.79, 1.63, 1.8 , 1.71, 1.  ]

### 40%
compression_factors = [1.  , 1.19, 1.54, 1.48, 1.46, 1.57, 1.66, 1.74, 1.79, 2.19, 2.34, 2.14,
 2.31, 2.34, 2.  , 1.89, 1.8 , 2.12, 2.64, 2.18, 1.98, 2.2 , 2.08, 1.  ]

### 40%

projections = {
    'query':        {'attr': r'self_attn\.q_proj',   'layers': list(range(3,24)), 'out_shape': [14, 64]},
    'key':          {'attr': r'self_attn\.k_proj',   'layers': list(range(3,24)), 'out_shape': [2, 64]},
    'value':        {'attr': r'self_attn\.v_proj',   'layers': list(range(15,24)), 'out_shape': [2, 64]},
    'o_proj':       {'attr': r'self_attn\.o_proj',   'layers': list(range(15,24)), 'in_shape':  [14, 64]},
    'mlp_up_proj':  {'attr': r'mlp\.up_proj',        'layers': list(range(7,24))},
    'mlp_down_proj':{'attr': r'mlp\.down_proj',      'layers': list(range(15,24))},
    'mlp_gate_proj':{'attr': r'mlp\.gate_proj',      'layers': list(range(7,24))},
}

config = {
    'defaults': ['/tensor_decomposition_base'],
    'rules': {}
}

for rule_name, spec in projections.items():
    for layer in spec['layers']:
        factor = compression_factors[layer]
        key = f"{rule_name}_layer_{layer}"
        pattern = rf"model\.layers\.{layer}\.{spec['attr']}"

        solver = {
            '_target_':           'percipio2.skiboot.td.config.Decomposition.solve_low_ecf',
            '_partial_':          True,
            'number_cores':       2,
            'compression_factor': factor,
            'min_ranks':          10,
            'topology':           '${oc.select:same_topology,btt}'
        }

        # deep‐copy so each rule gets its own list object
        if 'out_shape' in spec:
            solver['out_shape'] = copy.deepcopy(spec['out_shape'])
        if 'in_shape' in spec:
            solver['in_shape']  = copy.deepcopy(spec['in_shape'])

        config['rules'][key] = {
            'find':            {'pattern': pattern},
            'general_replace': {'solver_method': solver, 'decompose': True}
        }

class NoAliasDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True

out_path = "/home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/Qwen2.5-0.5B/td_cfg/qwen_0.5B_td_btt_bi_test.yaml"
with open(out_path, 'w') as f:
    # 1) Header comment + defaults block
    f.write("# @package model.config.percipio.tensor_decomposition\n\n")
    f.write("defaults:\n")
    f.write("  - /tensor_decomposition_base\n\n")
    # 2) Dump the rest (rules) with 2‐space indentation, no anchors
    yaml.dump(
        {'rules': config['rules']},
        f,
        Dumper=NoAliasDumper,
        sort_keys=False,
        indent=2
    )

print(f"Wrote updated config to {out_path}")


Wrote updated config to /home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/Qwen2.5-0.5B/td_cfg/qwen_0.5B_td_btt_bi_test.yaml


### Qwen3 8B

In [ ]:
import yaml
import copy
import os

### 10%
# compression_factors = [1.  , 1.19, 1.42, 1.21, 1.04, 1.09, 1.12, 1.19, 1.06, 1.01, 1.02, 1.09,
#  1.08, 1.15, 1.19, 1.3 , 1.39, 1.39, 1.36, 1.28, 1.51, 1.42, 1.23, 1.24,
#  1.27, 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ]

### 20%
# compression_factors = [1.  , 1.5 , 1.78, 1.51, 1.3 , 1.36, 1.4 , 1.49, 1.33, 1.26, 1.27, 1.37,
#  1.35, 1.45, 1.49, 1.63, 1.74, 1.74, 1.71, 1.6 , 1.9 , 1.78, 1.54, 1.55,
#  1.59, 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ]

### 30%
# compression_factors = [1.  , 2.01, 2.39, 2.03, 1.74, 1.82, 1.88, 2.  , 1.79, 1.69, 1.71, 1.84,
#  1.81, 1.94, 2.  , 2.18, 2.33, 2.33, 2.29, 2.14, 2.54, 2.38, 2.07, 2.08,
#  2.13, 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ]

### 40%
# compression_factors = [1.  , 2.94, 3.5 , 2.97, 2.55, 2.67, 2.75, 2.93, 2.62, 2.48, 2.5 , 2.69,
#  2.65, 2.84, 2.93, 3.19, 3.41, 3.42, 3.36, 3.14, 3.72, 3.49, 3.03, 3.04,
#  3.13, 1.2 , 1.03, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01]

### 50%
compression_factors = [1.08, 3.72, 4.43, 3.77, 3.23, 3.38, 3.48, 3.72, 3.32, 3.14, 3.17, 3.41,
 3.36, 3.6 , 3.7 , 4.04, 4.32, 4.33, 4.25, 3.98, 4.71, 4.42, 3.83, 3.85,
 3.96, 1.53, 1.31, 1.28, 1.28, 1.28, 1.28, 1.28, 1.28, 1.28, 1.28, 1.28]

layer_idx = list(range(0, 36))

projections = {
    'query':        {'attr': r'self_attn\.q_proj',   'layers': layer_idx, 'out_shape': [32, 128]},
    'key':          {'attr': r'self_attn\.k_proj',   'layers': layer_idx, 'out_shape': [8, 128]},
    'value':        {'attr': r'self_attn\.v_proj',   'layers': layer_idx, 'out_shape': [8, 128]},
    'o_proj':       {'attr': r'self_attn\.o_proj',   'layers': layer_idx, 'in_shape':  [32, 128]},
    'mlp_up_proj':  {'attr': r'mlp\.up_proj',        'layers': layer_idx},
    'mlp_down_proj':{'attr': r'mlp\.down_proj',      'layers': layer_idx},
    'mlp_gate_proj':{'attr': r'mlp\.gate_proj',      'layers': layer_idx},
}

config = {
    'defaults': ['/tensor_decomposition_base'],
    'rules': {}
}

for rule_name, spec in projections.items():
    for layer in spec['layers']:
        factor = compression_factors[layer]
        key = f"{rule_name}_layer_{layer}"
        pattern = rf"model\.layers\.{layer}\.{spec['attr']}"

        solver = {
            '_target_':           'percipio2.skiboot.td.config.Decomposition.solve_low_ecf',
            '_partial_':          True,
            'number_cores':       2,
            'compression_factor': factor,
            'min_ranks':          10,
            'topology':           '${oc.select:same_topology,btt}'
        }

        # deep‐copy so each rule gets its own list object
        if 'out_shape' in spec:
            solver['out_shape'] = copy.deepcopy(spec['out_shape'])
        if 'in_shape' in spec:
            solver['in_shape']  = copy.deepcopy(spec['in_shape'])

        config['rules'][key] = {
            'find':            {'pattern': pattern},
            'general_replace': {'solver_method': solver, 'decompose': True}
        }

class NoAliasDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True

out_path = "/home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/Qwen3-8B/td_cfg/qwen3_8B_td_btt_bi.yaml"
with open(out_path, 'w') as f:
    # 1) Header comment + defaults block
    f.write("# @package model.config.percipio.tensor_decomposition\n\n")
    f.write("defaults:\n")
    f.write("  - /tensor_decomposition_base\n\n")
    # 2) Dump the rest (rules) with 2‐space indentation, no anchors
    yaml.dump(
        {'rules': config['rules']},
        f,
        Dumper=NoAliasDumper,
        sort_keys=False,
        indent=2
    )

print(f"Wrote updated config to {out_path}")


Wrote updated config to /home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/Qwen3-8B/td_cfg/qwen3_8B_td_btt_bi.yaml


### Smollm 135M

In [ ]:
import yaml
import copy
import os

### 10% 102.58M
compression_factors = [1.  , 1.  , 1.94, 2.  , 1.77, 1.75, 1.58, 1.55, 1.55, 1.47, 1.3 , 1.13,
 1.01, 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ]

### 20% 91.8M
# compression_factors = [1.  , 1.66, 3.26, 3.35, 2.97, 2.93, 2.65, 2.59, 2.6 , 2.47, 2.17, 1.9 ,
#  1.69, 1.25, 1.02, 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
#  1.  , 1.  , 1.  , 1.  , 1.  , 1.  ]

### 30%
# compression_factors = [1.  , 2.82, 5.54, 5.7 , 5.06, 4.98, 4.5 , 4.41, 4.42, 4.19, 3.69, 3.23,
#  2.88, 2.12, 1.73, 1.56, 1.41, 1.18, 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ,
#  1.  , 1.  , 1.  , 1.  , 1.  , 1.  ]

### 40%
# compression_factors = [1.  , 3.72, 7.3 , 7.51, 6.67, 6.57, 5.93, 5.81, 5.82, 5.53, 4.87, 4.25,
#  3.79, 2.8 , 2.29, 2.05, 1.86, 1.55, 1.27, 1.21, 1.21, 1.21, 1.21, 1.21,
#  1.21, 1.21, 1.21, 1.21, 1.21, 1.21]


layer_list = list(range(1,30))

projections = {
    'query':        {'attr': r'self_attn\.q_proj',   'layers': layer_list, 'out_shape': [9, 64]},
    'key':          {'attr': r'self_attn\.k_proj',   'layers': layer_list, 'out_shape': [3, 64]},
    'value':        {'attr': r'self_attn\.v_proj',   'layers': layer_list, 'out_shape': [3, 64]},
    'o_proj':       {'attr': r'self_attn\.o_proj',   'layers': layer_list, 'in_shape':  [9, 64]},
    'mlp_up_proj':  {'attr': r'mlp\.up_proj',        'layers': layer_list},
    'mlp_down_proj':{'attr': r'mlp\.down_proj',      'layers': layer_list},
    'mlp_gate_proj':{'attr': r'mlp\.gate_proj',      'layers': layer_list},
}

config = {
    'defaults': ['/tensor_decomposition_base'],
    'rules': {}
}

for rule_name, spec in projections.items():
    for layer in spec['layers']:
        factor = compression_factors[layer]
        key = f"{rule_name}_layer_{layer}"
        pattern = rf"model\.layers\.{layer}\.{spec['attr']}"

        solver = {
            '_target_':           'percipio2.skiboot.td.config.Decomposition.solve_low_ecf',
            '_partial_':          True,
            'number_cores':       2,
            'compression_factor': factor,
            'min_ranks':          4,
            'topology':           '${oc.select:same_topology,btt}'
        }

        # deep‐copy so each rule gets its own list object
        if 'out_shape' in spec:
            solver['out_shape'] = copy.deepcopy(spec['out_shape'])
        if 'in_shape' in spec:
            solver['in_shape']  = copy.deepcopy(spec['in_shape'])

        config['rules'][key] = {
            'find':            {'pattern': pattern},
            'general_replace': {'solver_method': solver, 'decompose': True}
        }

class NoAliasDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True

out_path = "/home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/SmolLM-135M/td_cfg/smollm-135M_btt_bi_test.yaml"
with open(out_path, 'w') as f:
    # 1) Header comment + defaults block
    f.write("# @package model.config.percipio.tensor_decomposition\n\n")
    f.write("defaults:\n")
    f.write("  - /tensor_decomposition_base\n\n")
    # 2) Dump the rest (rules) with 2‐space indentation, no anchors
    yaml.dump(
        {'rules': config['rules']},
        f,
        Dumper=NoAliasDumper,
        sort_keys=False,
        indent=2
    )

print(f"Wrote updated config to {out_path}")


Wrote updated config to /home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/SmolLM-135M/td_cfg/smollm-135M_btt_bi_test.yaml


### take in a sample .yaml (not good)

In [2]:
from ruamel.yaml import YAML
import re, copy

def generate_layerwise_config(input_yaml_path: str,
                              output_yaml_path: str,
                              layer_ratios: dict[int, float]):
    """
    Reads a template YAML, clones each rule per layer with its own regex
    and compression_factor (breaking any shared references), then writes
    it out without any &id anchors or *id aliases.
    """
    yaml = YAML()
    yaml.preserve_quotes = True
    # disable alias/anchor emission entirely
    yaml.representer.ignore_aliases = lambda *args: True

    with open(input_yaml_path, 'r') as f:
        cfg = yaml.load(f)

    original_rules = cfg.get('rules', {})
    new_rules = {}

    for layer, ratio in layer_ratios.items():
        for rule_name, rule_body in original_rules.items():
            # deep-copy so no shared references
            new_rule = copy.deepcopy(rule_body)

            # 1) specialize the regex to this layer
            pat = new_rule['find']['pattern']
            new_rule['find']['pattern'] = re.sub(r'\([^)]+\)', str(layer), pat, count=1)

            # 2) inject per-layer ratio
            new_rule['general_replace']['solver_method']['compression_factor'] = ratio

            # 3) unique key
            new_rules[f"{rule_name}_layer{layer}"] = new_rule

    cfg['rules'] = new_rules

    with open(output_yaml_path, 'w') as f:
        yaml.dump(cfg, f)



# Cell 2: specify your paths & ratios, then run
template_path = "/home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/Qwen2.5-0.5B/td_cfg/qwen_0.5B_td_btt.yaml"
output_path   = "/home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/Qwen2.5-0.5B/td_cfg/qwen_0.5B_td_btt_bi_auto.yaml"

compression_factors = [1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.02, 1.25, 1.34, 1.22,
 1.32, 1.34, 1.14, 1.08, 1.03, 1.21, 1.51, 1.24, 1.13, 1.26, 1.19, 1.  ]

# derive the dict that the function expects:
layers = list(range(3, 24))  # [3,4,...,23]
layer_ratios = dict(zip(layers, compression_factors))

generate_layerwise_config(template_path, output_path, layer_ratios)
print(f"Wrote expanded config to {output_path}")



Wrote expanded config to /home/ubuntu/yequan/FLAT-LLM/ranks/wikitext2/Qwen2.5-0.5B/td_cfg/qwen_0.5B_td_btt_bi.yaml
